# BoltzGen → LigandMPNN → BoltzFold Pipeline

This notebook scaffolds the structure-validated de novo design pipeline:

1. **BoltzGen** generates de novo protein backbones conditioned on a target.
2. **LigandMPNN** designs sequences onto those backbones.
3. **BoltzFold** refolds each designed sequence to validate the structure.
4. **Filter & save** the best designs by structural agreement.

> **Status**: This notebook is a scaffold. `BoltzGenGenerator.generate()` runs the CLI but output parsing is pending Phase 5 — it returns an empty list. The MPNN and BoltzFold steps are guarded placeholders until upstream parsing lands. See `src/evedesign/models/boltzgen.py` for the integration code.

In [ ]:
import torch
from loguru import logger

from evedesign.system import System, Protein
from evedesign.models.boltzgen import BoltzGenGenerator

try:
    from evedesign.models.mpnn import LigandMPNN
    MPNN_AVAILABLE = True
except Exception as e:
    MPNN_AVAILABLE = False
    logger.warning(f"LigandMPNN not available: {e}")

try:
    from evedesign.models.boltzfold import BoltzFoldTransformer
    BOLTZFOLD_AVAILABLE = True
except Exception as e:
    BOLTZFOLD_AVAILABLE = False
    logger.warning(f"BoltzFoldTransformer not available: {e}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"torch version: {torch.__version__}")
print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"BoltzGen CLI available: {BoltzGenGenerator.available}")
print(f"MPNN available: {MPNN_AVAILABLE}")
print(f"BoltzFold available: {BOLTZFOLD_AVAILABLE}")

## Step 1 — Define target and binder

A `System` defines what BoltzGen designs *against* and what it designs *de novo*:

- **Target** (`rep='ACDEFGHIKLMNPQRSTVWY'`): a fixed sequence. BoltzGen treats this as the binding partner.
- **Binder** (`rep=None`, `min_length=60`, `max_length=100`): no sequence specified — BoltzGen generates a brand new backbone in this size range.

The combination `rep=None` + length range tells BoltzGen "design this entity de novo".

In [ ]:
target = Protein(
    rep="ACDEFGHIKLMNPQRSTVWY",
    id="target",
)
binder = Protein(
    rep=None,
    min_length=60,
    max_length=100,
    id="binder",
)
system = System([target, binder])

print(f"System has {len(system)} entities:")
for i, e in enumerate(system):
    seq_info = "rep=None" if e.rep is None else f"len(rep)={len(e.rep)}"
    length_info = (
        f"length={e.min_length}..{e.max_length}"
        if (e.min_length is not None or e.max_length is not None)
        else ""
    )
    print(f"  [{i}] id={e.id} type={e.type} {seq_info} {length_info}")

## Step 2 — Generate backbones with BoltzGen

Key parameters:

- `protocol="protein-anything"` — generic protein design (binders, scaffolds).
- `budget` — overall sampling budget (higher = more diverse designs, longer runtime). Set low here for a fast smoke test.
- `skip_inverse_folding=True` — skip BoltzGen's built-in MPNN-style sequence design step, since we run our own LigandMPNN downstream.
- `keep_tmp_dir=True` — preserve the boltzgen CLI output directory after `generate()` for inspection (useful while Phase 5 parsing is pending).

> **Note**: `generate()` currently returns `[]` because output parsing is not yet wired up (Phase 5). The CLI runs to completion and writes outputs to a temp directory; the path is logged for inspection.

In [ ]:
backbones = []
generator = BoltzGenGenerator(
    device=DEVICE,
    protocol="protein-anything",
    skip_inverse_folding=True,
    budget=2,
    keep_tmp_dir=True,
).build(system)

print(f"generator.ready     = {generator.ready}")
print(f"generator.available = {generator.available}")

if generator.available:
    try:
        backbones = generator.generate(num_designs=2)
        print(f"Got {len(backbones)} backbones back")
        if not backbones:
            print(
                "  (empty list — expected until Phase 5 parsing lands. "
                "Check the logged tmp directory for raw outputs.)"
            )
    except RuntimeError as e:
        print(f"BoltzGen CLI failed: {e}")
else:
    print(
        "Skipping generate() — boltzgen CLI not on PATH "
        "(install via `pip install evedesign[boltzgen]` on a CUDA-12 host)."
    )

## Step 3 — Sequence design with LigandMPNN (pending Phase 5)

Once BoltzGen returns `SystemInstance` objects with backbone coordinates, we'll run LigandMPNN to design sequences onto each backbone:

```python
mpnn = LigandMPNN(device=DEVICE).build(system)
designed = []
for backbone in backbones:
    instances = mpnn.generate(
        backbone, num_designs=4, temperature=0.1,
    )
    designed.extend(instances)
```

This step is gated on Phase 5 (BoltzGen output parsing). The cell below is a placeholder.

In [ ]:
designed = []
if backbones and MPNN_AVAILABLE:
    print(f"Would run LigandMPNN on {len(backbones)} backbones — pending Phase 5.")
else:
    reasons = []
    if not backbones:
        reasons.append("no backbones from BoltzGen")
    if not MPNN_AVAILABLE:
        reasons.append("LigandMPNN not importable")
    print(f"Skipping MPNN step ({', '.join(reasons)}).")

## Step 4 — Refold designed sequences with BoltzFold (pending Phase 5)

After LigandMPNN produces sequences, BoltzFold refolds each one to verify that the predicted structure stays close to the BoltzGen-generated backbone:

```python
folder = BoltzFoldTransformer(device=DEVICE).build(system)
refolded = folder.transform(designed)
```

The confidence scores (`iptm`, `complex_plddt`, `confidence_score`) and the predicted CIF for each refold land on the returned `SystemInstance` objects.

In [ ]:
refolded = []
if designed and BOLTZFOLD_AVAILABLE:
    print(f"Would refold {len(designed)} designs with BoltzFold — pending Phase 5.")
else:
    reasons = []
    if not designed:
        reasons.append("no designed sequences")
    if not BOLTZFOLD_AVAILABLE:
        reasons.append("BoltzFoldTransformer not importable")
    print(f"Skipping refold step ({', '.join(reasons)}).")

## Step 5 — Filter and save (pending Phase 5)

The final step ranks refolded designs by structural agreement (RMSD between the BoltzGen backbone and the BoltzFold refold) and confidence (`complex_plddt`), then writes the surviving designs to disk.

In [ ]:
print(
    f"Would rank and save {len(refolded)} refolded designs — pending Phase 5."
)

## Next steps when Phase 5 lands

1. Wire `BoltzGenGenerator.generate()` output parsing in `src/evedesign/models/boltzgen.py` so it returns real `SystemInstance` objects with backbones populated.
2. Verify the BoltzGen CIF outputs round-trip through `SystemInstance.serialize()` / `deserialize()`.
3. Replace the placeholder cells (Steps 3–5) with the real MPNN → BoltzFold → filter pipeline.
4. Add per-step `status_callback` wiring so progress is visible from the notebook UI.
5. Once stable, add a smoke test mirroring this notebook under `tests/`.